# Component 2 - Player Stats Ingestion

Runs `scripts/fetch_player_stats.py` step by step so the results can be inspected. The script holds the logic; this notebook only calls it. From the repo root you can also run it directly:

```
python scripts/fetch_player_stats.py --league nba
```

Outputs and the ESPN cache go to `model_data/`, which is gitignored. No modelling happens here.

## Settings

In [ ]:
LEAGUE = "nba"            # "nba" or "wnba"
SEASONS_BACK = 1          # previous seasons to include alongside the current one
SEASON_TYPES = (2,)       # 2 = regular season, 3 = playoffs
MIN_GAMES = 10            # players need at least this many games to get a prediction
LOOKAHEAD_HOURS = 48

In [ ]:
import sys
from pathlib import Path

# Find the repo's scripts/ folder whether the notebook runs from notebooks/ or the repo root.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "fetch_player_stats.py").exists())
sys.path.insert(0, str(root / "scripts"))

import importlib
import fetch_player_stats as fps
importlib.reload(fps)  # pick up script edits without restarting the kernel

ing = fps.Ingest(LEAGUE, SEASONS_BACK, SEASON_TYPES)
print("League:", LEAGUE.upper(), "| Seasons:", [ing.season_label(y) for y in ing.seasons], "| Today (ET):", ing.today)

## 1. Completed games and player game logs

In [ ]:
completed = ing.completed_games()
logs = ing.game_logs(completed)
logs.tail()

## 2. Upcoming games

In [ ]:
games = ing.upcoming_games(LOOKAHEAD_HOURS)
games

## 3. Players in upcoming games

In [ ]:
players = ing.upcoming_players(logs, games, MIN_GAMES)
players.head(20)

## 4. Odds name check

In [ ]:
unmatched = ing.odds_name_check(logs)

## 5. Save

In [ ]:
ing.save(logs, games, players)